# Rectangle packing with native QAOA

All 19 source stages, the 4 by 2 container and three rectangles, 17 active variables, 102 regenerated Ising terms, ten QAOA layers, 60 objective queries and 10,000 actual shots per query and final readout.

Native local COBYLA and the nonconstant-coefficient norm schedule reproduce the optimization setup at the original sizes. Finite sampling uses fixed local seeds. The final reported packing is a word observed in the final sample, and its cost is checked against the separate exact geometry table. Query traces describe the native run.

## Load the native packing workflow

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.model import (
    HEIGHT,
    PENALTY,
    RECTANGLES,
    WIDTH,
    constraint_cost,
    make_model,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.plots import (
    save_costs,
    save_packing,
    save_trace,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.quantum import (
    LAYERS,
    program_builder,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.workflow import (
    MAX_QUERIES,
    SHOTS,
    initial_angles,
    run_workflow,
)

folder = Path("artifacts/development/rectangle-packing")


## Keep the original container and rectangle sizes

In [ ]:
print("Container and rectangles:", WIDTH, HEIGHT, RECTANGLES)


## Define the original placement constraints

In [ ]:
print("Each rectangle appears at most once, with disjoint cells and valid bounds.")


## Regenerate the fresh model

In [ ]:
model = make_model()


## Inspect bounded variables and terms

In [ ]:
print("Placement variables and Ising terms:", len(model.placements), len(model.terms))


## Keep all ten layers and the original penalty

In [ ]:
print("Layers and penalty:", LAYERS, PENALTY)


## Apply the finite optimizer cap

In [ ]:
print("Maximum objective queries:", MAX_QUERIES)


## Construct the original-size ansatz

In [ ]:
build = program_builder(model)


## Retain the actual sampling budget

In [ ]:
print("Shots per histogram:", SHOTS)


## Synthesize the initial norm-ramped circuit

In [ ]:
initial_program = build(initial_angles(model))
print("Initial resources:", initial_program.report["resources"])


## Run finite-sample mean optimization

In [ ]:
result = run_workflow()
print(
    "Actual objective queries and shots:",
    len(result.evaluations),
    len(result.evaluations) * SHOTS,
)
print("Final draws:", sum(count for _, count in result.sampled))
print("Best sampled mean:", result.selected.mean_cost)
print("Budget stop and convergence:", result.budget_reached, result.converged)


## Plot the observed optimizer trace

In [ ]:
display(Image(filename=str(save_trace(result, folder / "trace.png"))))


## Use the actual final sample

In [ ]:
assert len(result.sampled) > 0


## Plot the cost histogram

In [ ]:
display(Image(filename=str(save_costs(result, folder / "costs.png"))))


## Choose the best observed placement word

In [ ]:
print("Observed word and cost:", result.word, float(result.model.cost[result.word]))


## Preserve the fixed out-of-bounds variables as zero

In [ ]:
print("Active variables:", result.model.placements)


## Read the selected nonzero placements

In [ ]:
print("Observed placements:", result.placements)


## Draw the observed rectangle layout

In [ ]:
display(Image(filename=str(save_packing(result, folder / "packing.png"))))


## Check the geometry and exact reference

In [ ]:
print("Independent minimum cost:", result.exact_cost)
assert constraint_cost(result.placements) == float(result.model.cost[result.word])
print("Native resources:", result.program.report["resources"])
